# PinchBench RQ Figures

This notebook loads `results/*.json`, computes RQ-aligned metrics, and exports paper-ready figures with a unified visual style.

- Output directory: `analysis/paper_figures/`
- Formats: PNG and PDF
- Compatibility: supports both legacy single-attempt results and newer retry-aware results
- Safety: when the current slice cannot support an RQ, the notebook exports a placeholder figure instead of drawing a misleading chart


In [ ]:
from __future__ import annotations

import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

RESULT_GLOB = "results/*.json"
INCLUDE_FILES: list[str] = []
MODEL_FILTER: str | None = None
TASK_FILTER: str | None = None
FOCUS_RESULT: str | None = "results/0025_minimax-cn-MiniMax-M2-5.json"
OUTPUT_DIR = Path("analysis/paper_figures")
SAVE_FORMATS = ("png", "pdf")

STYLE = {
    "font.size": 14,
    "axes.titlesize": 18,
    "axes.labelsize": 15,
    "axes.linewidth": 1.1,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 11,
    "figure.titlesize": 18,
    "savefig.dpi": 220,
    "axes.grid": True,
    "grid.alpha": 0.18,
    "grid.linewidth": 0.8,
    "axes.facecolor": "white",
    "figure.facecolor": "white",
    "font.sans-serif": ["DejaVu Sans", "Noto Sans CJK SC", "SimHei", "Arial Unicode MS"],
    "axes.unicode_minus": False,
}
mpl.rcParams.update(STYLE)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PALETTE = ["#1f3a5f", "#3e7cb1", "#ff7f11", "#2a9d8f", "#d1495b", "#6c757d"]
SERIES_FIELDS = ["model", "feedback_policy", "feedback_format", "context_policy"]
print(f"Figures will be written to: {OUTPUT_DIR.resolve()}")


In [ ]:
def pick_paths() -> list[Path]:
    if INCLUDE_FILES:
        paths = [Path(p) for p in INCLUDE_FILES]
    else:
        paths = sorted(Path().glob(RESULT_GLOB))
    return [p for p in paths if p.exists()]


def normalize_retry_policies(result: dict) -> dict:
    retry = result.get("retry_policies") or {}
    if retry:
        return {
            "feedback_policy": retry.get("feedback_policy", "unknown"),
            "feedback_format": retry.get("feedback_format", "unknown"),
            "context_policy": retry.get("context_policy", "unknown"),
            "stop_rule": retry.get("stop_rule", "unknown"),
        }
    return {
        "feedback_policy": "legacy-single-turn",
        "feedback_format": "legacy",
        "context_policy": "legacy",
        "stop_rule": "legacy",
    }


def synthesize_attempts(task: dict) -> list[dict]:
    attempts = list(task.get("attempts") or [])
    if attempts:
        return attempts
    completion = task.get("completion") or {}
    grading = task.get("grading") or {}
    score = completion.get("score", grading.get("mean", 0.0))
    max_score = completion.get("max_score", grading.get("max", 1.0) or 1.0)
    return [{
        "attempt": 1,
        "execution": {
            "usage": task.get("usage") or {},
            "execution_time": task.get("execution_time", 0.0),
        },
        "grading": {
            "score": score,
            "max_score": max_score,
            "notes": (completion or {}).get("notes", ""),
        },
        "feedback_prompt": None,
        "session_reset": False,
        "workspace_restored": False,
    }]


def infer_first_success_attempt(attempts: list[dict]) -> int | None:
    for attempt in attempts:
        grading = attempt.get("grading") or {}
        score = float(grading.get("score", 0.0) or 0.0)
        max_score = float(grading.get("max_score", 0.0) or 0.0)
        if max_score > 0 and score >= max_score:
            return int(attempt.get("attempt", 0) or 0)
    return None


def passed_from_attempts(attempts: list[dict], completion: dict) -> bool:
    if isinstance(completion, dict) and completion.get("passed") is not None:
        return bool(completion.get("passed"))
    return infer_first_success_attempt(attempts) is not None


def make_series_label(row: pd.Series) -> str:
    return " | ".join(str(row[field]) for field in SERIES_FIELDS)


def load_frames(paths: list[Path]) -> tuple[pd.DataFrame, pd.DataFrame]:
    task_rows: list[dict] = []
    attempt_rows: list[dict] = []

    for path in paths:
        result = json.loads(path.read_text(encoding="utf-8"))
        retry = normalize_retry_policies(result)
        configured_attempts = int(result.get("max_task_attempts") or 0)
        model = result.get("model", "unknown")
        run_id = result.get("run_id", path.stem)

        for task in result.get("tasks") or []:
            frontmatter = task.get("frontmatter") or {}
            task_id = task.get("task_id") or frontmatter.get("id") or "unknown-task"
            attempts = synthesize_attempts(task)
            attempts = sorted(attempts, key=lambda item: int(item.get("attempt", 0) or 0))
            observed_attempts = max((int(a.get("attempt", 0) or 0) for a in attempts), default=1)
            configured = configured_attempts or observed_attempts
            first_success_attempt = task.get("first_success_attempt")
            if first_success_attempt is None:
                first_success_attempt = infer_first_success_attempt(attempts)
            completion = task.get("completion") or {}
            sample_id = f"{path.stem}::{task_id}"

            task_row = {
                "sample_id": sample_id,
                "source_file": path.name,
                "run_id": run_id,
                "model": model,
                "task_id": task_id,
                "task_name": frontmatter.get("name", task_id),
                "category": frontmatter.get("category", "unknown"),
                "grading_type": frontmatter.get("grading_type", "unknown"),
                "timeout_seconds": float(frontmatter.get("timeout_seconds", 0.0) or 0.0),
                "workspace_file_count": len(frontmatter.get("workspace_files") or []),
                "feedback_policy": retry["feedback_policy"],
                "feedback_format": retry["feedback_format"],
                "context_policy": retry["context_policy"],
                "stop_rule": retry["stop_rule"],
                "configured_attempt_budget": configured,
                "observed_attempts": observed_attempts,
                "first_success_attempt": first_success_attempt,
                "passed": passed_from_attempts(attempts, completion),
                "stop_reason": task.get("stop_reason", "unknown"),
                "final_score": float(completion.get("score", 0.0) or 0.0),
                "final_max_score": float(completion.get("max_score", 1.0) or 1.0),
                "timed_out": bool(task.get("timed_out", False)),
            }
            task_row["series_label"] = make_series_label(pd.Series(task_row))
            task_rows.append(task_row)

            cumulative_tokens = 0.0
            cumulative_cost = 0.0
            cumulative_time = 0.0
            for attempt in attempts:
                attempt_no = int(attempt.get("attempt", 0) or 0)
                execution = attempt.get("execution") or {}
                grading = attempt.get("grading") or {}
                usage = execution.get("usage") or {}
                tokens = float(usage.get("total_tokens", 0.0) or 0.0)
                cost = float(usage.get("cost_usd", 0.0) or 0.0)
                time_s = float(execution.get("execution_time", 0.0) or 0.0)

                cumulative_tokens = float(
                    ((execution.get("cumulative_usage") or {}).get("total_tokens"))
                    or (cumulative_tokens + tokens)
                )
                cumulative_cost = float(
                    ((execution.get("cumulative_usage") or {}).get("cost_usd"))
                    or (cumulative_cost + cost)
                )
                cumulative_time = float(
                    execution.get("cumulative_execution_time")
                    or (cumulative_time + time_s)
                )

                score = float(grading.get("score", 0.0) or 0.0)
                max_score = float(grading.get("max_score", 1.0) or 1.0)
                passed = bool(max_score > 0 and score >= max_score)

                attempt_rows.append({
                    **task_row,
                    "attempt": attempt_no,
                    "score": score,
                    "max_score": max_score,
                    "normalized_score": score / max_score if max_score > 0 else np.nan,
                    "attempt_tokens": tokens,
                    "attempt_cost_usd": cost,
                    "attempt_time_seconds": time_s,
                    "cumulative_tokens": cumulative_tokens,
                    "cumulative_cost_usd": cumulative_cost,
                    "cumulative_time_seconds": cumulative_time,
                    "passed_this_attempt": passed,
                    "session_reset": bool(attempt.get("session_reset", False)),
                    "workspace_restored": bool(attempt.get("workspace_restored", False)),
                    "feedback_prompt_chars": len((attempt.get("feedback_prompt") or "")),
                })

    task_df = pd.DataFrame(task_rows)
    attempt_df = pd.DataFrame(attempt_rows)

    if MODEL_FILTER and not task_df.empty:
        keep = task_df["model"] == MODEL_FILTER
        task_df = task_df.loc[keep].copy()
        attempt_df = attempt_df.loc[attempt_df["sample_id"].isin(task_df["sample_id"])].copy()
    if TASK_FILTER and not task_df.empty:
        keep = task_df["task_id"] == TASK_FILTER
        task_df = task_df.loc[keep].copy()
        attempt_df = attempt_df.loc[attempt_df["sample_id"].isin(task_df["sample_id"])].copy()

    return task_df.sort_values(["series_label", "task_id", "sample_id"]), attempt_df.sort_values(["series_label", "sample_id", "attempt"])


paths = pick_paths()
if not paths:
    raise SystemExit(f"No result files matched: {RESULT_GLOB}")

task_df, attempt_df = load_frames(paths)
print(f"Loaded {len(paths)} files, {task_df['sample_id'].nunique()} task-runs, {attempt_df.shape[0]} attempts")
task_df.head(3)


In [ ]:
coverage = {
    "files": len(paths),
    "task_runs": int(task_df["sample_id"].nunique()) if not task_df.empty else 0,
    "unique_tasks": int(task_df["task_id"].nunique()) if not task_df.empty else 0,
    "models": int(task_df["model"].nunique()) if not task_df.empty else 0,
    "policy_variants": int(task_df[["feedback_policy", "feedback_format", "context_policy"]].drop_duplicates().shape[0]) if not task_df.empty else 0,
    "max_observed_attempts": int(task_df["observed_attempts"].max()) if not task_df.empty else 0,
    "successful_runs": int(task_df["passed"].sum()) if not task_df.empty else 0,
}

rq_status = {
    "RQ1 marginal benefit": bool(not task_df.empty and task_df["observed_attempts"].max() >= 2),
    "RQ2 attempts-to-success": bool(not task_df.empty and task_df["first_success_attempt"].notna().any()),
    "RQ3 token/cost tradeoff": bool(not attempt_df.empty and task_df["observed_attempts"].max() >= 2),
    "RQ4 policy/context comparison": bool(not task_df.empty and task_df[["feedback_policy", "feedback_format", "context_policy"]].drop_duplicates().shape[0] >= 2),
}

print("Coverage summary")
for key, value in coverage.items():
    print(f"- {key}: {value}")
print("\nRQ readiness")
for key, value in rq_status.items():
    print(f"- {key}: {'ready' if value else 'insufficient variation'}")

display(task_df[["source_file", "model", "task_id", "configured_attempt_budget", "observed_attempts", "feedback_policy", "feedback_format", "context_policy", "passed", "first_success_attempt", "stop_reason"]].head(12))


In [ ]:
def save_figure(fig: plt.Figure, stem: str) -> None:
    fig.subplots_adjust(left=0.12, right=0.98, top=0.9, bottom=0.17)
    for ext in SAVE_FORMATS:
        fig.savefig(OUTPUT_DIR / f"{stem}.{ext}", bbox_inches="tight", pad_inches=0.03)
    plt.close(fig)


def base_axes(ax: plt.Axes, title: str, xlabel: str, ylabel: str, ylim=None) -> None:
    ax.set_title(title, pad=10)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if ylim is not None:
        ax.set_ylim(*ylim)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(True, axis="y")


def placeholder_figure(stem: str, title: str, message: str) -> None:
    fig, ax = plt.subplots(figsize=(6.6, 4.0))
    ax.axis("off")
    ax.text(0.5, 0.62, title, ha="center", va="center", fontsize=18, fontweight="bold")
    ax.text(0.5, 0.42, message, ha="center", va="center", fontsize=13, wrap=True)
    save_figure(fig, stem)


def compute_curves(task_frame: pd.DataFrame, attempt_frame: pd.DataFrame) -> pd.DataFrame:
    curve_rows: list[dict] = []
    if task_frame.empty:
        return pd.DataFrame(curve_rows)

    for series_label, group in task_frame.groupby("series_label"):
        max_k = int(group["observed_attempts"].max())
        for k in range(1, max_k + 1):
            eligible = group[group["observed_attempts"] >= k].copy()
            if eligible.empty:
                continue
            eligible = eligible.assign(
                success_by_k=eligible["first_success_attempt"].fillna(np.inf) <= k,
                success_by_prev=eligible["first_success_attempt"].fillna(np.inf) <= (k - 1),
            )
            latest = (
                attempt_frame[
                    attempt_frame["sample_id"].isin(eligible["sample_id"]) & (attempt_frame["attempt"] <= k)
                ]
                .sort_values(["sample_id", "attempt"])
                .groupby("sample_id", as_index=False)
                .tail(1)
            )
            curve_rows.append({
                "series_label": series_label,
                "k": k,
                "eligible_runs": int(eligible.shape[0]),
                "success_rate": float(eligible["success_by_k"].mean()),
                "marginal_gain": float(eligible["success_by_k"].mean() - eligible["success_by_prev"].mean()),
                "avg_cumulative_tokens": float(latest["cumulative_tokens"].mean()),
                "avg_cumulative_cost_usd": float(latest["cumulative_cost_usd"].mean()),
                "avg_cumulative_time_seconds": float(latest["cumulative_time_seconds"].mean()),
            })
    return pd.DataFrame(curve_rows)


curve_df = compute_curves(task_df, attempt_df)
curve_df.head(10)


In [ ]:
def plot_lines(df: pd.DataFrame, x: str, y: str, stem: str, title: str, xlabel: str, ylabel: str, ylim=None) -> None:
    if df.empty:
        placeholder_figure(stem, title, "No eligible samples for this figure.")
        return
    fig, ax = plt.subplots(figsize=(6.6, 4.0))
    labels = list(df["series_label"].drop_duplicates())
    for idx, label in enumerate(labels):
        sub = df[df["series_label"] == label].sort_values(x)
        color = PALETTE[idx % len(PALETTE)]
        ax.plot(sub[x], sub[y], marker="o", linewidth=2.4, markersize=6.8, color=color, label=label)
    base_axes(ax, title, xlabel, ylabel, ylim=ylim)
    ax.legend(loc="best", frameon=False)
    save_figure(fig, stem)


if curve_df.empty or curve_df["k"].max() < 2:
    placeholder_figure(
        "fig_rq1_success_at_k",
        "RQ1: Success@k",
        "Current slice has no multi-attempt evidence. Add runs with k >= 2 to estimate marginal retry benefit.",
    )
    placeholder_figure(
        "fig_rq1_marginal_gain_by_attempt",
        "RQ1: Marginal Gain by Attempt",
        "Current slice has no multi-attempt evidence. Add runs with k >= 2 to estimate attempt-level gains.",
    )
else:
    plot_lines(
        curve_df,
        x="k",
        y="success_rate",
        stem="fig_rq1_success_at_k",
        title="RQ1: Success@k",
        xlabel="Attempt budget (k)",
        ylabel="Success rate",
        ylim=(0, 1.02),
    )
    plot_lines(
        curve_df,
        x="k",
        y="marginal_gain",
        stem="fig_rq1_marginal_gain_by_attempt",
        title="RQ1: Marginal Gain by Attempt",
        xlabel="Attempt number",
        ylabel="Delta success rate",
        ylim=(min(-0.02, float(curve_df['marginal_gain'].min()) - 0.02), max(0.05, float(curve_df['marginal_gain'].max()) + 0.05)),
    )


success_df = task_df[task_df["first_success_attempt"].notna()].copy()
if success_df.empty:
    placeholder_figure(
        "fig_rq2_attempts_to_success",
        "RQ2: Attempts to Success",
        "No successful runs in the current slice, so the first-success distribution cannot be estimated yet.",
    )
else:
    fig, axes = plt.subplots(1, 2, figsize=(6.6, 4.0))
    pmf = success_df.groupby("first_success_attempt").size().sort_index()
    pmf = pmf / pmf.sum()
    axes[0].bar(pmf.index.astype(int), pmf.values, color=PALETTE[1], width=0.72)
    base_axes(axes[0], "First-success PMF", "Attempt", "Share of successes", ylim=(0, max(0.35, float(pmf.max()) + 0.08)))
    cdf = pmf.cumsum()
    axes[1].plot(cdf.index.astype(int), cdf.values, marker="o", linewidth=2.4, color=PALETTE[2])
    base_axes(axes[1], "First-success CDF", "Attempt", "P(success by k)", ylim=(0, 1.02))
    save_figure(fig, "fig_rq2_attempts_to_success")


if curve_df.empty or curve_df["k"].max() < 2:
    placeholder_figure(
        "fig_rq3_tokens_vs_success",
        "RQ3: Tokens vs Success",
        "Current slice has no retry curve, so token-efficiency tradeoffs cannot be estimated yet.",
    )
    placeholder_figure(
        "fig_rq3_cost_vs_success",
        "RQ3: Cost vs Success",
        "Current slice has no retry curve, so cost-efficiency tradeoffs cannot be estimated yet.",
    )
else:
    plot_lines(
        curve_df,
        x="avg_cumulative_tokens",
        y="success_rate",
        stem="fig_rq3_tokens_vs_success",
        title="RQ3: Tokens vs Success",
        xlabel="Average cumulative tokens per task",
        ylabel="Success rate",
        ylim=(0, 1.02),
    )
    plot_lines(
        curve_df,
        x="avg_cumulative_cost_usd",
        y="success_rate",
        stem="fig_rq3_cost_vs_success",
        title="RQ3: Cost vs Success",
        xlabel="Average cumulative cost per task (USD)",
        ylabel="Success rate",
        ylim=(0, 1.02),
    )


policy_variants = task_df[["feedback_policy", "feedback_format", "context_policy"]].drop_duplicates()
if policy_variants.shape[0] < 2:
    placeholder_figure(
        "fig_rq4_policy_comparison",
        "RQ4: Policy Comparison",
        "Only one policy/context variant is present in the current slice. Add policy-controlled runs before comparing convergence.",
    )
else:
    group_max = task_df.groupby("series_label")["observed_attempts"].max()
    common_k = int(group_max.min())
    latest = curve_df[curve_df["k"] == common_k].copy().sort_values("success_rate", ascending=False)
    fig, axes = plt.subplots(1, 2, figsize=(6.6, 4.0))
    y = np.arange(latest.shape[0])
    axes[0].barh(y, latest["success_rate"], color=PALETTE[0], alpha=0.95)
    axes[0].set_yticks(y, latest["series_label"])
    base_axes(axes[0], f"Success at common k={common_k}", "Success rate", "Series", ylim=(0, latest.shape[0]))
    axes[0].grid(True, axis="x")
    axes[1].barh(y, latest["avg_cumulative_tokens"], color=PALETTE[2], alpha=0.95)
    axes[1].set_yticks(y, latest["series_label"])
    base_axes(axes[1], f"Token cost at common k={common_k}", "Average cumulative tokens", "Series", ylim=(0, latest.shape[0]))
    axes[1].grid(True, axis="x")
    save_figure(fig, "fig_rq4_policy_comparison")

print(f"Exported figures to: {OUTPUT_DIR.resolve()}")


In [ ]:
summary_rows = []
if not curve_df.empty:
    for series_label, sub in curve_df.groupby("series_label"):
        final = sub.sort_values("k").iloc[-1]
        summary_rows.append({
            "series_label": series_label,
            "max_k": int(final["k"]),
            "eligible_runs_at_final_k": int(final["eligible_runs"]),
            "success_rate_at_final_k": round(float(final["success_rate"]), 4),
            "avg_tokens_at_final_k": round(float(final["avg_cumulative_tokens"]), 1),
            "avg_cost_usd_at_final_k": round(float(final["avg_cumulative_cost_usd"]), 6),
        })
summary_df = pd.DataFrame(summary_rows).sort_values(["success_rate_at_final_k", "avg_tokens_at_final_k"], ascending=[False, True]) if summary_rows else pd.DataFrame()
summary_df.to_csv(OUTPUT_DIR / "rq_summary_table.csv", index=False)
display(summary_df)


In [ ]:
if FOCUS_RESULT and Path(FOCUS_RESULT).exists():
    focus_task_df, focus_attempt_df = load_frames([Path(FOCUS_RESULT)])
    if not focus_attempt_df.empty:
        target = focus_attempt_df.sort_values(["sample_id", "attempt"]).copy()
        title_stub = Path(FOCUS_RESULT).stem
        fig, axes = plt.subplots(1, 2, figsize=(6.6, 4.0))
        for sample_id, sub in target.groupby("sample_id"):
            label = sub.iloc[0]["task_id"]
            axes[0].plot(sub["attempt"], sub["normalized_score"], marker="o", linewidth=2.4, label=label)
            axes[1].plot(sub["attempt"], sub["cumulative_tokens"], marker="o", linewidth=2.4, label=label)
        base_axes(axes[0], "Score trajectory", "Attempt", "Normalized score", ylim=(0, 1.02))
        base_axes(axes[1], "Token trajectory", "Attempt", "Cumulative tokens")
        axes[0].legend(loc="best", frameon=False)
        save_figure(fig, f"fig_appendix_attempt_trajectory_{title_stub}")
        print(f"Exported appendix trajectory for: {FOCUS_RESULT}")
    else:
        print(f"No attempts found in focus result: {FOCUS_RESULT}")
else:
    print("FOCUS_RESULT not set or file missing; skipped appendix trajectory.")


## Interpretation Notes

- `success@k` and cost curves only use runs that were actually observed up to attempt `k`; they do not treat missing higher-budget runs as failures.
- Legacy JSON files without retry metadata are labeled as `legacy-single-turn | legacy | legacy`.
- When there is only one policy variant or no successful run, the notebook exports a placeholder figure. This is intentional and keeps the paper narrative honest.
- For RQ4, the comparison is made at the largest attempt budget shared by all policy variants in the filtered slice.
